# Module 7 Assignment: Rebuild Your Persona Agent on the OpenAI Agents SDK

Lab 6 rebuilt Study Buddy on top of the OpenAI Agents SDK: same persona, same notes, same three
tools from Lab 5, but with `Agent` and `Runner` replacing the hand written loop and dispatcher.

This assignment makes you do the same move on your **own** Persona Agent. By the end of Module 6
your agent had a persona, study notes, and three tools routed by a `call_tool(name, arguments)`
dispatcher you wrote by hand. Nothing about what your agent *does* needs to change here: you are
porting it to the framework, not redesigning it.

**What you'll build:**
1. An Azure configured `OpenAIChatCompletionsModel`, with tracing disabled, exactly like Lab 6
2. Your persona and notes, carried over from your Module 6 assignment
3. All three of your existing tools, each ported to `@function_tool`
4. One `Agent` that declares your persona, your notes, and all three tools
5. A test using `await Runner.run`
6. The same Gradio chat interface every lab and assignment has used since Lab 4, adapted from Lab 6

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 6 made the same kind of change to each of Study
Buddy's three tools, and do the analogous thing to yours.

## 0. Install the packages (run once)

Same as Lab 6.

In [ ]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

## 1. Imports

**TODO:** Copy the imports from Lab 6, Step 1 (including `gradio as gr`), plus `requests` (and
anything else your own tools from the Module 6 assignment need, for example `html` if one of your
tools calls an API that returns HTML-escaped text).

In [ ]:
import os
import json
import requests
from pypdf import PdfReader
import gradio as gr
from openai import AsyncAzureOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, function_tool, set_tracing_disabled

## 2. Load your Azure OpenAI credentials from `.env`

Identical to Lab 6 and every assignment before it.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from Lab 6, Step 3, unchanged: call `set_tracing_disabled(True)`, build an
`AsyncAzureOpenAI` client from your endpoint, key, and API version, then wrap it and your deployment
name in `OpenAIChatCompletionsModel`.

In [ ]:
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(model=DEPLOYMENT, openai_client=azure_client)

## 4. Your persona and notes

**TODO:** Bring over your persona description and your study notes from your Module 6 assignment,
unchanged. If your notes lived in a PDF, reuse the same `load_notes` helper from Lab 6, Step 4.

In [ ]:
# Persona and notes carried over unchanged from the Module 6 assignment
persona = (
    "You are Professor Byte, a friendly, encouraging study coach. You help a student review "
    "the material in their course notes, explain ideas in plain language with short examples, "
    "and keep answers concise. You stay patient and upbeat, and you are honest when something "
    "is not covered in the notes."
)

notes = ""
if os.path.exists("notes/notes.pdf"):
    reader = PdfReader("notes/notes.pdf")
    for page in reader.pages:
        text = page.extract_text()
        if text:
            notes += text
else:
    print("WARNING: notes/notes.pdf not found - copy your Module 6 notes.pdf into the notes folder.")

print(persona)
print(notes[:300], "...")

## 5. Port your first tool

**TODO:** Take your first tool from the Module 6 assignment and rewrite it as a plain, typed Python
function decorated with `@function_tool`, the same way Lab 6, Step 5a ported
`log_missed_question_tool`. Keep the function's actual behavior identical: only the decorator and
the removal of your hand written JSON schema should change.

In [ ]:
# First tool (from Module 5/6): log a question the notes can't answer, so it can be followed up later.
unknown_questions = []

@function_tool
def record_unknown_question(question: str) -> str:
    """Always use this tool to record any question you could not answer from the notes.

    Args:
        question: The question that could not be answered.
    """
    unknown_questions.append(question)
    print(f"[tool] record_unknown_question -> {question!r}")
    return f"Recorded. {len(unknown_questions)} unanswered question(s) logged so far."

## 6. Port your second tool

**TODO:** Same idea, for your second tool (in Module 6 this was the one backed by a free, no-key
external API), the same way Lab 6, Step 5b ported `fetch_bonus_question_tool`.

In [ ]:
# Second tool: look up a word with the Free Dictionary API (https://dictionaryapi.dev) - free, no key.
@function_tool
def define_word(word: str) -> str:
    """Look up the dictionary definition of a single English word using an online dictionary.

    Args:
        word: The single English word to define.
    """
    print(f"[tool] define_word -> {word!r}")
    word = str(word).strip()
    if not word:
        return "No word was given to define."
    url = f"https://api.dictionaryapi.dev/api/v2/entries/en/{requests.utils.quote(word)}"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 404:
            return f"No dictionary definition found for '{word}'."
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.RequestException:
        return "The dictionary service is unavailable right now. Please try again in a moment."
    except ValueError:
        return "The dictionary service sent back a response I couldn't read."

    try:
        meaning = data[0]["meanings"][0]
        part_of_speech = meaning.get("partOfSpeech", "")
        definition = meaning["definitions"][0]["definition"]
    except (IndexError, KeyError, TypeError):
        return f"No dictionary definition found for '{word}'."
    return f"{word} ({part_of_speech}): {definition}"

## 7. Port your third tool

**TODO:** Same idea, for your third, self-designed tool, the same way Lab 6, Step 5c ported
`check_progress_tool`.

In [ ]:
# Third tool: bookmark a topic the student wants to come back to, and show the review list.
bookmarked_topics = []

@function_tool
def bookmark_topic(topic: str) -> str:
    """Bookmark a topic the student wants to review later, and return the current review list.

    Args:
        topic: A short name for the topic to review later.
    """
    print(f"[tool] bookmark_topic -> {topic!r}")
    topic = str(topic).strip()
    if not topic:
        return "No topic was given to bookmark."
    if topic.lower() in (t.lower() for t in bookmarked_topics):
        return f"'{topic}' is already bookmarked. Review list: {', '.join(bookmarked_topics)}"
    bookmarked_topics.append(topic)
    return f"Bookmarked '{topic}'. Review list ({len(bookmarked_topics)}): {', '.join(bookmarked_topics)}"

## 8. Build your Agent

**TODO:** Build one `Agent`, the same way Lab 6, Step 6 built Study Buddy: a `name`, `instructions`
built from your persona and your notes (with the same rules your old system prompt had about when
to use each tool), a `tools` list containing all three of your ported tools, and `model=azure_model`.

Notice what is missing compared to your Module 6 assignment: no `call_tool(name, arguments)`
dispatcher, and no JSON schema for any of the three tools. `Agent` and `@function_tool` already
cover both.

In [ ]:
instructions = f"""
{persona}

You are helping a student study. Answer their questions using the notes below whenever you can.

## Notes:
{notes}

## Tools:
If the student asks something you cannot answer from the notes, use the record_unknown_question tool to record it, then tell them it has been logged.
If the student asks what a specific word means, or you use a term they may not know, use the define_word tool to get its dictionary definition.
If the student says they are confused by a topic or wants to come back to it later, use the bookmark_topic tool to add it to their review list.

Stay in character as described above.
"""

professor_byte = Agent(
    name="Professor Byte",
    instructions=instructions,
    tools=[record_unknown_question, define_word, bookmark_topic],
    model=azure_model,
)

print([tool.name for tool in professor_byte.tools])

## 9. Test it: `await Runner.run`

**TODO:** Run your agent with `await Runner.run(your_agent, "...")`, the same way Lab 6, Step 7
tested Study Buddy. Pick a message that should trigger at least one of your three tools, and
confirm the tool's visible effect (a line appended to a file, a number that changed, or whatever
your tool does) actually happened.

Stick to `await Runner.run(...)` here, not `Runner.run_sync`: Jupyter notebooks already run their
own event loop in the background, and `Runner.run_sync` raises
`RuntimeError: AgentRunner.run_sync() cannot be called when an event loop is already running.` when
called from a notebook cell.

In [ ]:
print("Bookmarks before:", bookmarked_topics)

result = await Runner.run(
    professor_byte,
    "I'm really confused by recursion. Can you bookmark it so I come back to it later?",
)
print(result.final_output)

# Confirm the tool's visible effect actually happened
print("Bookmarks after:", bookmarked_topics)

## 10. Give it a real chat UI with Gradio

**TODO:** Adapt the `chat` function and `gr.ChatInterface` call from Lab 6, Step 8 to your own
agent: an `async def chat(message, history)` function that rebuilds `history` as a plain list of
`{"role": ..., "content": ...}` dictionaries, appends the new `message` as one more `user` turn,
passes that to `await Runner.run(your_agent, ...)`, and returns `result.final_output`, handed to
`gr.ChatInterface(chat).launch(inbrowser=True)`.

As in Lab 6, `gr.ChatInterface` accepts your async `chat` function directly, and its `history`
already comes in the shape `Runner.run` expects, so no extra conversion is needed.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [ ]:
async def chat(message, history):
    # Rebuild history as plain {"role", "content"} dicts (Gradio can attach extra keys
    # and sometimes content-part lists), then add the new message as one more user turn.
    messages = []
    for h in history:
        content = h["content"]
        if isinstance(content, list):
            content = " ".join(
                part.get("text", "") if isinstance(part, dict) else str(part) for part in content
            )
        messages.append({"role": h["role"], "content": content})
    messages.append({"role": "user", "content": message})

    result = await Runner.run(professor_byte, messages)
    return result.final_output


gr.ChatInterface(chat).launch(inbrowser=True)

## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. Which parts of your Module 6 assignment did you copy over completely unchanged? Which parts,
   if any, needed a real change rather than just a different decorator?
2. Compare your old `call_tool(name, arguments)` dispatcher to what `Agent` and `Runner` do for you
   now. What exactly is `Runner.run` doing in place of that dispatcher and your `while`/`for` loop?
3. Your Module 6 system prompt included sentences telling the model when to use each tool. Did any
   of that wording need to change for the OpenAI Agents SDK, or did it carry over directly into
   `instructions`?
4. If you added a fourth tool to this agent, what would you need to write, and what would you not
   need to write, compared to adding a fourth tool by hand back in Module 6?

## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm all three of your ported tools are reachable, `tools=[...]` lists all three, and at least
  one tool call actually happened during your test in Step 9 or Step 10.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module7Assignment_JaneDoe.ipynb`, then submit
  it.

## Reflection Answers

**1. What carried over unchanged, and what needed a real change?**
My persona, the `notes/notes.pdf` loading code, the Azure credentials cell, and the bodies of all three tools (`record_unknown_question`, `define_word`, `bookmark_topic`) carried over unchanged. The rules in my system prompt also moved into `instructions` word for word. The only edits to the tools were adding `@function_tool`, type hints (`question: str`, `-> str`), and docstrings. The docstrings hold the descriptions that used to live in my JSON schemas. The parts that really changed were the client and the loop. `AzureOpenAI` became `AsyncAzureOpenAI` wrapped in `OpenAIChatCompletionsModel`. `call_tool`, the `tools` list of JSON schemas, and my `while`/`for` loop were all deleted and replaced by one `Agent` and `await Runner.run(...)`. Because `Runner.run` is async, `chat` became `async def chat`.

**2. What does `Runner.run` do in place of my dispatcher and loop?**
In Module 6, my `chat()` sent the messages to the model and then looped `while finish_reason == "tool_calls"`. For each tool call, it parsed the JSON arguments, used `call_tool`'s `if`/`elif` chain to pick the Python function by name, appended the result as a `"tool"` message with the matching `tool_call_id`, and called the model again. `Runner.run` does that whole cycle for me. It sends the instructions, conversation, and tool schemas to the model. When the model asks for a tool, it finds the matching `@function_tool` by name in the agent's `tools` list, validates and parses the arguments, runs the function, and feeds the result back. It keeps going until the model gives a final answer, which it returns as `result.final_output`.

**3. Did the tool-usage wording need to change?**
No. The three sentences telling the model when to use each tool carried over directly into `instructions`. That works because the model still gets the same kind of input: text instructions plus tool names and descriptions. The SDK only changes how that input is packaged and how tool calls are executed, not how the model decides which tool to call. The only wording that moved was each tool's description, which went from the JSON `"description"` field to the function's docstring.

**4. What would adding a fourth tool take now, compared to Module 6?**
I would still write the Python function itself, give it type hints and a docstring, decorate it with `@function_tool`, add it to `tools=[...]` in the `Agent`, and add one sentence to `instructions` saying when to use it. I would not need to write a JSON schema, because `@function_tool` generates it from the signature and docstring. I also would not need a new `elif` branch in a dispatcher, and the loop would not change, because `Runner.run` routes by tool name automatically. In Module 6, the same tool needed the function, a hand-written schema dict, an entry in the `tools` list, a new `elif` in `call_tool`, and the system prompt sentence.
